
# XGBoost Regression — End‑to‑End Tutorial

**Goal:** Learn how XGBoost works for regression, from intuition and math to training, tuning, and interpreting a model with real data.

**What you'll do:**
1. Understand how gradient boosting with trees works (the XGBoost way).
2. Train a strong baseline model with scikit‑learn API (`XGBRegressor`).
3. Add **early stopping**, evaluate with **RMSE/MAE/R²**, and do **hyperparameter tuning**.
4. Inspect **feature importance** and **SHAP explanations**.
5. Use **monotonic constraints**, handle **missing values**, and save/load models.



## 0) Environment Setup

> If running locally, execute the next cell to install dependencies. (Skip on platforms that already have them.)


In [ ]:

# If needed (e.g., on Colab or a fresh environment), uncomment:
# !pip install -q xgboost scikit-learn shap matplotlib pandas numpy



## 1) XGBoost for Regression — Intuition & Math

**Big picture:** XGBoost is a **gradient boosting** algorithm that builds decision trees **sequentially**. Each new tree tries to fix the residual errors of the model so far, using **gradient (and Hessian) information** of a chosen loss function.

### Objective
We minimize the **regularized** objective:
\begin{align}
\mathcal{L}^{(t)} = \sum_{i=1}^n \ell\big(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\big) + \Omega(f_t),
\end{align}
where \(f_t\) is the new tree added at boosting round \(t\), and \(\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2\) penalizes the number of leaves \(T\) and their weights \(w_j\).

Using a second‑order Taylor expansion around \(\hat{y}_i^{(t-1)}\):
\begin{align}
\ell(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) \approx \ell(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \tfrac{1}{2} h_i f_t(x_i)^2,
\end{align}
where \(g_i = \frac{\partial \ell}{\partial \hat{y}} \), \(h_i = \frac{\partial^2 \ell}{\partial \hat{y}^2}\).

If a leaf \(j\) contains data \(I_j\), the **optimal leaf weight** is:
\begin{align}
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}.
\end{align}

And the **split gain** (improvement from splitting a node) is:
\begin{align}
\text{Gain} = \frac{1}{2}\left[
\frac{(\sum g_L)^2}{\sum h_L + \lambda} +
\frac{(\sum g_R)^2}{\sum h_R + \lambda} -
\frac{(\sum g_T)^2}{\sum h_T + \lambda}
\right] - \gamma,
\end{align}
where \(L, R, T\) are the left child, right child, and the parent (total), and \(\gamma\) is the minimum loss reduction required to make a split.

**Key properties:**
- **Second‑order boosting** (uses gradients & Hessians)
- **Sparsity‑aware** (handles missing values by learning a default direction)
- **Regularized** trees (L2 via `reg_lambda`, L1 via `reg_alpha`)
- Fast histogram‑based tree building (`tree_method="hist"`), optional GPU (`gpu_hist`)
- Level‑wise growth (good generalization; contrasts with LightGBM’s leaf‑wise growth)



## 2) Dataset: California Housing (Regression)

We'll predict median house values from the classic California Housing dataset.


In [ ]:

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target  # in 100k USD

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train.shape, X_valid.shape, X_test.shape



## 3) Train a Strong Baseline with `XGBRegressor` + Early Stopping


In [ ]:

from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="rmse",
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=100,
    early_stopping_rounds=50
)

best_n_estimators = xgb.best_iteration + 1 if hasattr(xgb, "best_iteration") else xgb.n_estimators
best_n_estimators


In [ ]:

def evaluate(model, X_tr, y_tr, X_te, y_te, name="Model"):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    rmse_tr = mean_squared_error(y_tr, pred_tr, squared=False)
    rmse_te = mean_squared_error(y_te, pred_te, squared=False)
    mae_te = mean_absolute_error(y_te, pred_te)
    r2_te = r2_score(y_te, pred_te)
    print(f"{name}:")
    print(f"  Train RMSE: {rmse_tr:.4f}")
    print(f"  Test  RMSE: {rmse_te:.4f}")
    print(f"  Test   MAE: {mae_te:.4f}")
    print(f"  Test    R²: {r2_te:.4f}")
    return {"rmse_train": rmse_tr, "rmse_test": rmse_te, "mae_test": mae_te, "r2_test": r2_te}

metrics = evaluate(xgb, X_train, y_train, X_test, y_test, name="XGBoost (early stop)")
metrics



## 4) Feature Importance (Gain & Permutation)

Beware: built‑in importances can be biased. Use permutation importance (or SHAP) for more reliable signals.


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Built-in importance
importance = xgb.get_booster().get_score(importance_type="gain")
items = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)
labels = [k for k, _ in items]
values = [v for _, v in items]

plt.figure(figsize=(8, 4))
plt.bar(range(len(values)), values)
plt.xticks(range(len(values)), labels, rotation=45, ha="right")
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()



## 5) SHAP for Local & Global Explanations

SHAP explains individual predictions and provides consistent global importance. (May take a moment.)


In [ ]:

import shap

# TreeExplainer is optimized for tree models
explainer = shap.TreeExplainer(xgb)
# Use a subset for speed
X_sample = X_test.sample(n=min(200, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

# Summary plot
shap.summary_plot(shap_values, X_sample, show=True)



## 6) Hyperparameter Tuning (RandomizedSearchCV)

We’ll search over a compact, high‑leverage space. Always keep **early stopping** when possible.


In [ ]:

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "n_estimators": randint(400, 2000),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_lambda": uniform(0.0, 2.0),
    "reg_alpha": uniform(0.0, 1.0),
    "min_child_weight": randint(1, 10),
}

base = XGBRegressor(
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse",
)

search = RandomizedSearchCV(
    estimator=base,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    verbose=1,
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", (-search.best_score_):.4f)
best_model = search.best_estimator_
evaluate(best_model, X_train, y_train, X_test, y_test, name="Best XGB (CV)")



## 7) Monotonic Constraints (When domain logic demands it)

If you **know** the target should increase with \(x_1\) and decrease with \(x_2\), enforce it:

- `monotone_constraints=(1, -1, 0, ...)` for each feature


In [ ]:

# Synthetic data to demonstrate monotonic constraints
rng = np.random.RandomState(0)
n = 2000
Xsyn = pd.DataFrame({
    "price": rng.uniform(1, 10, size=n),      # should increase y
    "discount": rng.uniform(0, 1, size=n),    # should decrease y
    "noise": rng.normal(size=n)
})
y_syn = 3.0*Xsyn["price"] - 2.0*Xsyn["discount"] + 0.3*Xsyn["noise"] + rng.normal(scale=0.3, size=n)

X_tr, X_te, y_tr, y_te = train_test_split(Xsyn, y_syn, test_size=0.3, random_state=42)

xgb_mono = XGBRegressor(
    tree_method="hist",
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    monotone_constraints=(1, -1, 0),  # +price, -discount, free noise
    random_state=42
)
xgb_mono.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

evaluate(xgb_mono, X_tr, y_tr, X_te, y_te, name="XGB with Monotonic Constraints")



## 8) Handling Missing Values & Categoricals

- **Missing values:** XGBoost learns a default split direction; you can pass `np.nan` directly—no imputation required.
- **Categoricals:** Prefer one‑hot encoding for low cardinality. For high cardinality, consider target/impact encoding with proper CV to avoid leakage. Recent XGBoost versions also support native categorical splits (check your version docs).


In [ ]:

# Example: passing NaNs directly
X_nan = X.copy()
X_nan.loc[X_nan.sample(frac=0.05, random_state=1).index, X_nan.columns[0]] = np.nan

xgb_nan = XGBRegressor(
    tree_method="hist",
    n_estimators=600,
    random_state=42
)
xgb_nan.fit(X_nan, y, eval_set=[(X_nan, y)], verbose=False)
print("Model trained with NaNs present (default directions learned).")



## 9) Save & Load Models


In [ ]:

# Option A: Native XGBoost format
xgb.save_model("xgb_model.json")

# Option B: joblib/pickle (scikit-learn API)
import joblib
joblib.dump(xgb, "xgb_model.joblib")
print("Saved xgb_model.json and xgb_model.joblib")



## 10) Practical Recipe & Tuning Tips

1. **Start simple:** `tree_method="hist"`, `max_depth=6`, `n_estimators=2000`, `learning_rate=0.05`.
2. **Always use** a validation set and **early stopping** (e.g., `early_stopping_rounds=50`).
3. **Control complexity:** increase `min_child_weight`, use `subsample`/`colsample_bytree`, and add `reg_lambda`/`reg_alpha`.
4. **Tune by buckets:** try depths 3–10; shrink `learning_rate` and increase `n_estimators` if underfitting.
5. **Check leakage:** especially with time series (use time‑based splits) and with encoding schemes.
6. **Monitor metrics:** RMSE/MAE for scale, R² for proportion explained; keep a simple baseline for context.
7. **Explainability:** Use **SHAP** to validate that drivers make sense; consider **monotone constraints** when you have domain monotonicities.



## Appendix: Native `xgb.cv` with DMatrix


In [ ]:

import xgboost as xgb
dtrain = xgb.DMatrix(X_train, label=y_train)
params = {
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "eval_metric": "rmse",
    "seed": 42,
}
cv = xgb.cv(
    params,
    dtrain,
    num_boost_round=2000,
    nfold=5,
    early_stopping_rounds=50,
    verbose_eval=False,
)
cv.tail()
